In [1]:
import pandas as pd
import numpy as np

# World Happiness Report 2023 — representative data
# Source: https://www.kaggle.com/datasets/ajaypalsinghlo/world-happiness-report-2023

df = pd.read_csv('world_happiness_2023.csv')
df.columns = ['Country','Region','Happiness_Score','GDP','Social_Support',
              'Life_Expectancy','Freedom','Generosity','Corruption']


print(f"Dataset: {len(df)} countries, {len(df.columns)} columns")
df.head()

Dataset: 63 countries, 9 columns


,Country,Region,Happiness_Score,GDP,Social_Support,Life_Expectancy,Freedom,Generosity,Corruption
0,Finland,Western Europe,7.804,10.775,0.954,71.9,0.949,0.142,0.179
1,Denmark,Western Europe,7.586,10.933,0.954,72.7,0.931,0.168,0.234
2,Iceland,Western Europe,7.525,10.878,0.983,72.5,0.961,0.260,0.150
3,Israel,Middle East and North Africa,7.473,10.527,0.916,72.4,0.903,0.149,0.826
4,Netherlands,Western Europe,7.464,11.015,0.939,72.4,0.879,0.240,0.296


In [2]:
import plotly.express as px
import plotly.graph_objects as go

# Explore the dataset before you start
print("Regions in dataset:")
print(df['Region'].value_counts())
print("\nScore range:", df['Happiness_Score'].min(), "–", df['Happiness_Score'].max())
print("\nBottom 10 countries:")
print(df.nsmallest(10, 'Happiness_Score')[['Country','Region','Happiness_Score']])

Regions in dataset:
Region
Western Europe                  15
Latin America and Caribbean     13
Central and Eastern Europe       7
Sub-Saharan Africa               7
Middle East and North Africa     6
North America and ANZ            4
Southeast Asia                   4
South Asia                       4
East Asia                        3
Name: count, dtype: int64

Score range: 1.859 – 7.804

Bottom 10 countries:
        Country                        Region  Happiness_Score
60  Afghanistan                    South Asia            1.859
61      Lebanon  Middle East and North Africa            2.392
62     Zimbabwe            Sub-Saharan Africa            2.995
52     Ethiopia            Sub-Saharan Africa            3.564
53     Tanzania            Sub-Saharan Africa            3.698
48   Bangladesh                    South Asia            3.892
47        India                    South Asia            4.036
50        Kenya            Sub-Saharan Africa            4.112
54       Uganda

In [3]:
# Task 1: Regional comparison bar chart
# -------------------------------------

# Step 1: Compute average happiness score by region
region_avg = (df.groupby('Region')['Happiness_Score']
              .mean()
              .reset_index()
              .sort_values('Happiness_Score'))  # sort for horizontal bar

print(region_avg)

colors = ['#BBBBBB'] * len(region_avg)
colors[0] = '#D55E00'          # first row = lowest region = highlight

fig = go.Figure(go.Bar(
    x=region_avg['Happiness_Score'], y=region_avg['Region'], orientation='h',
    marker_color=colors,
    text=region_avg['Happiness_Score'].round(1), textposition='outside'))
fig.update_layout(
    title='South Asia is the least happy region, scoring about half of Western Europe',
    xaxis_title='Average happiness score (0–10)', yaxis_title='',
    xaxis_range=[0, 8.5], template='simple_white', width=900, height=450)
fig.update_xaxes(showgrid=False)
fig.show()

                         Region  Happiness_Score
5                    South Asia         3.618250
7            Sub-Saharan Africa         4.064714
3  Middle East and North Africa         4.943333
6                Southeast Asia         5.695250
2   Latin America and Caribbean         5.699000
1                     East Asia         5.966000
0    Central and Eastern Europe         6.338143
4         North America and ANZ         7.018250
8                Western Europe         7.085533


In [4]:
# Task 2: Top 8 vs. Bottom 8 contrast
# ------------------------------------

# Step 1: Get top and bottom countries
top8 = df.nlargest(8, 'Happiness_Score').copy()
top8['Group'] = 'Top 8'
bottom8 = df.nsmallest(8, 'Happiness_Score').copy()
bottom8['Group'] = 'Bottom 8'

combined = pd.concat([bottom8, top8]).sort_values('Happiness_Score')
global_avg = df['Happiness_Score'].mean()
print(f"Global average: {global_avg:.2f}")

gap = top8['Happiness_Score'].mean() / bottom8['Happiness_Score'].mean()

fig = px.bar(combined, x='Happiness_Score', y='Country', color='Group', orientation='h',
    color_discrete_map={'Top 8': '#0072B2', 'Bottom 8': '#D55E00'},
    labels={'Happiness_Score': 'Happiness score (0–10)', 'Country': ''},
    title=f'The happiest countries score about {gap:.1f}x higher than the least happy',
    template='simple_white', width=900, height=600)
fig.add_vline(x=global_avg, line_dash='dash', line_color='grey',
    annotation_text=f'Global average {global_avg:.1f}', annotation_position='top')
fig.update_xaxes(range=[0, 8.5], showgrid=False)
fig.update_yaxes(categoryorder='total ascending')
fig.update_layout(legend_title='')
fig.show()

Global average: 5.81


In [5]:
regions = ['Western Europe', 'Latin America and Caribbean', 'East Asia',
           'Sub-Saharan Africa', 'South Asia']
g = df[df['Region'].isin(regions)].groupby('Region')[['Freedom', 'Social_Support']].mean().reset_index()
long = g.melt(id_vars='Region', var_name='Factor', value_name='Score')

fig = px.bar(long, x='Region', y='Score', color='Factor', barmode='group',
    color_discrete_map={'Freedom': '#0072B2', 'Social_Support': '#E69F00'},
    title='Western Europe leads on freedom and social support; South Asia and Sub-Saharan Africa trail',
    template='simple_white', width=900, height=450)
fig.update_yaxes(range=[0, 1])
fig.show()